# TP1 Recommender Systems

Victor Bouvier d'Acher  
Geoffroy Rodriguez  
Arezki Meriane  

In [458]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error


## Part 1 : Manipulating Data with Pandas library

    -

In [459]:
df_movie = pd.read_csv("movies_metadata.csv")

small_df = df_movie[["title", "release_date", "budget", "revenue", "runtime", "genres"]]

/tmp/ipykernel_8948/2710135394.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_movie = pd.read_csv("movies_metadata.csv")


In [460]:
small_df.head()

,title,release_date,budget,revenue,runtime,genres
0,Toy Story,1995-10-30,30000000,373554033.0,81.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,Jumanji,1995-12-15,65000000,262797249.0,104.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,Grumpier Old Men,1995-12-22,0,0.0,101.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
3,Waiting to Exhale,1995-12-22,16000000,81452156.0,127.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
4,Father of the Bride Part II,1995-02-10,0,76578911.0,106.0,"[{'id': 35, 'name': 'Comedy'}]"


Le budget correspond au coût de production du film

    -

In [461]:
small_df.dtypes

title            object
release_date     object
budget           object
revenue         float64
runtime         float64
genres           object
dtype: object

La donnée "budget" est de type "object", c'est à dire qu'il y a tous types de données dans ces colonnes (str, float, int, etc...).

    -

In [462]:
# en commentaire pour éviter l'erreur, décomenter et relancer tout le notebook pour la constater 
# small_df["budget"] = small_df["budget"].astype("Float64")

ValueError: could not convert string to float: '/ff9qCepilowshEtG2GYWwzt2bs4.jpg'

En changeant le type de budget en float avec *astype*, on obtient l'erreur suivante: **could not convert string to float: '/ff9qCepilowshEtG2GYWwzt2bs4.jpg'**  
cette erreur est causé car il n'est pas possible de convertir un string en un float.  

    -

In [463]:
def to_float(number):
    """
    Vérifie si l'object peut etre converti en float sinon retourne NaN
    """
    try:
        return float(number)
    except:
        return np.NaN


    -

In [464]:
small_df["budget"] = small_df["budget"].apply(to_float)

/tmp/ipykernel_8948/2306316853.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  small_df["budget"] = small_df["budget"].apply(to_float)


In [465]:
print(small_df.dtypes)

title            object
release_date     object
budget          float64
revenue         float64
runtime         float64
genres           object
dtype: object



Le type de "budget" est devenu **float**

    -

In [466]:
small_df["release_date"].head()

0    1995-10-30
1    1995-12-15
2    1995-12-22
3    1995-12-22
4    1995-02-10
Name: release_date, dtype: object

In [467]:
small_df["release_date"] = pd.to_datetime(small_df["release_date"], errors='coerce')

/tmp/ipykernel_8948/2883416495.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  small_df["release_date"] = pd.to_datetime(small_df["release_date"], errors='coerce')


In [468]:
small_df["year"] = small_df["release_date"].dt.year

/tmp/ipykernel_8948/3896245949.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  small_df["year"] = small_df["release_date"].dt.year


In [469]:
df_year = small_df.sort_values(by="year")

In [470]:
print(df_year.iloc[0])

title                              Passage of Venus
release_date                    1874-12-09 00:00:00
budget                                          0.0
revenue                                         0.0
runtime                                         1.0
genres          [{'id': 99, 'name': 'Documentary'}]
year                                         1874.0
Name: 34940, dtype: object


Le film le plus ancien est **Passage of Venus** qui date de **1874**

In [471]:
df_revenue = small_df.sort_values(by="revenue", ascending=False)

In [472]:
print(df_revenue.iloc[0])

title                                                      Avatar
release_date                                  2009-12-10 00:00:00
budget                                                237000000.0
revenue                                              2787965087.0
runtime                                                     162.0
genres          [{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...
year                                                       2009.0
Name: 14551, dtype: object


Le film le plus rentable est **Avatar** avec un revenu de **2 787 965 087**

In [473]:
new = small_df[small_df["revenue"] >= 1000000000] 

In [474]:
new.head()

,title,release_date,budget,revenue,runtime,genres,year
1639,Titanic,1997-11-18,200000000.0,1.845034e+09,194.0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",1997.0
7000,The Lord of the Rings: The Return of the King,2003-12-01,94000000.0,1.118889e+09,201.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",2003.0
11008,Pirates of the Caribbean: Dead Man's Chest,2006-06-20,200000000.0,1.065660e+09,151.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",2006.0
12481,The Dark Knight,2008-07-16,185000000.0,1.004558e+09,152.0,"[{'id': 18, 'name': 'Drama'}, {'id': 28, 'name...",2008.0
14551,Avatar,2009-12-10,237000000.0,2.787965e+09,162.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",2009.0


In [475]:
new2 = new[new["budget"] <= 150000000]

In [476]:
new2.head()

,title,release_date,budget,revenue,runtime,genres,year
7000,The Lord of the Rings: The Return of the King,2003-12-01,94000000.0,1.118889e+09,201.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",2003.0
17437,Harry Potter and the Deathly Hallows: Part 2,2011-07-07,125000000.0,1.342000e+09,130.0,"[{'id': 10751, 'name': 'Family'}, {'id': 14, '...",2011.0
22110,Frozen,2013-11-27,150000000.0,1.274219e+09,102.0,"[{'id': 16, 'name': 'Animation'}, {'id': 12, '...",2013.0
25084,Jurassic World,2015-06-09,150000000.0,1.513529e+09,124.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",2015.0
30700,Minions,2015-06-17,74000000.0,1.156731e+09,91.0,"[{'id': 10751, 'name': 'Family'}, {'id': 16, '...",2015.0


## Part 2 : Building a simple recommeder

In [477]:
def score_poids(v, R, C, m:int=100):

    return ((v/(v+m))*R + (m/(m+v)*C))


In [478]:
df_naif = df_movie[["title", "runtime", "vote_average", "vote_count"]]
m = 100

In [479]:
df_naif = df_naif[df_naif["runtime"] >= 45]
df_naif = df_naif[df_naif["runtime"] <= 300]
df_naif = df_naif[df_naif["vote_count"] > m]
df_naif = df_naif[df_naif["vote_average"] >= df_naif['vote_average'].quantile(0.8)]

Nous avons défini le nombre minimum de votes **m = 100** pour qu'un film soit pris en compte.  
Prendre une valeur trop faible peut surrévaluer des films trop peu noté et une valeur trop forte peut ignoré certain film avec une popularité importante.  
Nous avons jugé qu'une valeur de **100** était un bon compromis.  
Nous avons aussi appliqué des filtres suplémentaires au dataframe pour ne prendre en compte que les films éligibles.

In [480]:
titre = "Toy Story"

print(score_poids(df_naif[df_naif["title"] == titre]["vote_count"], 
        df_naif[df_naif["title"] == titre]["vote_average"],
        np.mean(df_naif["vote_average"])))

0    7.697115
dtype: float64


    1) Ci-dessus, nous calculons le score du film Toy Story grace à notre fonction score_poids. Celui-ci réspecte les conditions pour recevoir une note et obtient 7,69.

    2)

In [481]:
moyenne = np.mean(df_naif["vote_average"])

df_naif['score'] = df_naif.apply(
    lambda x: score_poids( 
        x[3], 
        x[2],
        moyenne,
        m=m
        ), 
        axis=1
    )

In [482]:
df_naif.head()

,title,runtime,vote_average,vote_count,score
0,Toy Story,81.0,7.7,5415.0,7.697115
5,Heat,170.0,7.7,1886.0,7.691989
15,Casino,178.0,7.8,1343.0,7.782045
16,Sense and Sensibility,136.0,7.2,364.0,7.273472
28,The City of Lost Children,108.0,7.6,308.0,7.585517


    3)

In [483]:
df_naif = df_naif.sort_values("score", ascending=False)

    4)

In [484]:
df_naif.head()

,title,runtime,vote_average,vote_count,score
10309,Dilwale Dulhania Le Jayenge,190.0,9.1,661.0,8.895126
314,The Shawshank Redemption,142.0,8.5,8358.0,8.488661
834,The Godfather,175.0,8.5,6024.0,8.484339
40251,Your Name.,106.0,8.5,1030.0,8.415125
12481,The Dark Knight,152.0,8.3,12269.0,8.293863


## Part 3 : Implement a user-based collaborative filtering system

In [485]:
df_user = pd.read_csv("ratings_small.csv")

In [486]:
df_user.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


    1)

In [487]:
df_user_NA = df_user.pivot_table(
                        index='userId',
                        columns='movieId', 
                        values='rating'
                    )

In [488]:
df_user_NA.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,161084,161155,161594,161830,161918,161944,162376,162542,162672,163949
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


    2)

Il y a beaucoup de valeurs manquantes car les utilisateurs notent que quelques uns des 163949 films du dataframe.  

On va remplir les valeurs manquantes de la matrice avec des 0 car le 0 permet d'indiquer :

    - soit que l'utilisateur n'a pas apprécié le film
    - soit que l'utilisateur ne connaît pas le film (ou n'a pas encore donné son opinion)

Cette méthode à des avantages :

    - Elle est simple et rapide,
    - On peut appliquer des algorithmes de machine learning ne gérant pas bien les matrices creuses,

Mais aussi des inconvénients :
 
    - On considère donc que de ne pas donner d'avis est équivalent à ne pas apprécier le film, ce qui peut ne pas être vrai,
    - On prend en compte les films qui n'ont pas été notés du tout,
    - Cela fait baisser les moyennes générales des notes et réduit donc l'écart général entre les moyennes des films bien notés et ceux mal notés.


In [489]:
df_user_rating = df_user_NA.fillna(0)

In [490]:
df_user_rating.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,161084,161155,161594,161830,161918,161944,162376,162542,162672,163949
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


    3)

In [491]:
df_user_similarity = pd.DataFrame(cosine_similarity(df_user_rating))
df_user_similarity.index += 1
df_user_similarity.columns += 1

In [492]:
df_user_similarity.head()

,1,2,3,4,5,6,7,8,9,10,...,662,663,664,665,666,667,668,669,670,671
1,1.000000,0.000000,0.000000,0.074482,0.016818,0.000000,0.083884,0.000000,0.012843,0.000000,...,0.000000,0.000000,0.014474,0.043719,0.000000,0.000000,0.000000,0.062917,0.000000,0.017466
2,0.000000,1.000000,0.124295,0.118821,0.103646,0.000000,0.212985,0.113190,0.113333,0.043213,...,0.477306,0.063202,0.077745,0.164162,0.466281,0.425462,0.084646,0.024140,0.170595,0.113175
3,0.000000,0.124295,1.000000,0.081640,0.151531,0.060691,0.154714,0.249781,0.134475,0.114672,...,0.161205,0.064198,0.176134,0.158357,0.177098,0.124562,0.124911,0.080984,0.136606,0.170193
4,0.074482,0.118821,0.081640,1.000000,0.130649,0.079648,0.319745,0.191013,0.030417,0.137186,...,0.114319,0.047228,0.136579,0.254030,0.121905,0.088735,0.068483,0.104309,0.054512,0.211609
5,0.016818,0.103646,0.151531,0.130649,1.000000,0.063796,0.095888,0.165712,0.086616,0.032370,...,0.191029,0.021142,0.146173,0.224245,0.139721,0.058252,0.042926,0.038358,0.062642,0.225086


In [493]:
def find_nnn(userId, n, mat):
    sim = mat[userId].sort_values(ascending=False).drop(userId)

    return sim.head(n)

In [494]:
n_nn = find_nnn(1, 5, df_user_similarity)
print(n_nn)

325    0.371852
634    0.194093
341    0.162819
310    0.157524
207    0.152746
Name: 1, dtype: float64


    4)

In [495]:
def film_recommendation(userId, n, k, mat, df):
    nn = find_nnn(userId=userId, n=n, mat=mat)

    user_film = df.loc[userId]

    film_non_vu = user_film[user_film == 0].index


    df_nn = df.loc[nn.index, film_non_vu]

    poids_film = df_nn.T.dot(nn) / nn.sum()


    film_recommender = poids_film.sort_values(ascending=False)

    return film_recommender.head(k)

La fonction calcul d'abord les users les plus proches de l'utilisateur choisi avec la similarité cosinus   
puis il ne prend que les films non nôté de l'utilisateur choisi avec les notes des users les plus proches de l'utilisateur choisi.  
puis il calcul la note de chaque film en faisant un produit scalaire entre les notes et la similarité cosinus des users les plus proches de l'utilisateur choisi divisé par la somme de la similarité cosinus des users les plus proches de l'utilisateur choisi.  
il tri le resultat par ordre decroissant et renvoie que les k film les mieux nôté.  

In [496]:
film_a_voir = film_recommendation(2, 6, 10, df_user_similarity, df_user_rating)

In [497]:
print(film_a_voir)

movieId
318    3.994984
597    3.169322
21     3.156852
595    3.011133
344    2.997948
316    2.825532
380    2.818385
32     2.667113
434    2.658439
329    2.497791
dtype: float64


In [498]:
def trouver_n(mat, df, taille_n):

    erreur = []

    for n in range(1, taille_n):
        
        score_predi = []
        score_reel = []

        for user_id in df.index:

            user_film = df.loc[user_id]

            film_vu = user_film[user_film != 0]

            nn = find_nnn(userId=user_id, n=n, mat=mat)


            df_nn = df.loc[nn.index, film_vu.index]

            poids_film = df_nn.T.dot(nn) / nn.sum()

            score_predi.extend(poids_film)
            score_reel.extend(film_vu.values)


        rmse = np.sqrt(mean_squared_error(score_predi, score_reel))
        erreur.append((n, rmse))

    meilleur = min(erreur, key=lambda x: x[1])
    return meilleur



Cette procedure permet de calculer le n optimal pour notre algorithme de recommandation.  
Il effectue les memes taches que la fonction **film_recommendation** à la différence qu'on ne note pas les films non nôté de l'utilisateur choisi mais ceux qui ont été nôté par celui-ci.  
Ils seront comparer resultat de l'agorithme en appliquant la RMSE sur le resultat.  
Le plus petit score sera retenu pour etre le nombre **n** optimale.

In [500]:
n_optimal = trouver_n(df_user_similarity, df_user_rating, 20)

In [501]:
print(n_optimal)

(6, 2.4979062888186476)


Le meilleur nombre de n est **6** avec un RMSE de **2.5**  

C'est un résultat plutot mauvais car les notes sont sur 5, ceux qui veut dire que les notes sont en generale 2.5 points a côté de la note donnée par l'utilisateur.  
La raison de cette différence est dû au choix de remplissage des NA, comme les valeurs manquantes ont été remplacés par des 0, Les moyennes des films auront tendances à être sous nôté par rapport a la normal et à la note donnée par l'utilisateur.  